<a href="https://colab.research.google.com/github/Casperbieglecki/IT480-FA26/blob/main/Assignments/Assignment%202/Assignment2_TransferLearning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Assignment 2: Transfer Learning Across Two Datasets

**Name:** *Casper Bieglecki*

**Prerequisite:** Transfer Learning lecture (frozen feature extraction)

This assignment uses frozen transfer learning, exactly what was covered in lecture: load a pretrained base, freeze it, add a new head, train only that. You'll do this twice, on two different datasets, and compare the results.

Data loading is provided for you below. Everything after that, building the model, training, evaluating, and reflecting, is yours to write, following the general instructions in each section.

**Submission:** GitHub repo URL and this notebook's URL on Canvas.

## Setup — Imports

In [1]:
import tensorflow as tf
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os

from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense
from tensorflow.keras.models import Model
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

---
# Part 1: Fire/Smoke Detection

## 1A — Load the Data

In [3]:
!pip install legacy-cgi --quiet
!pip install opendatasets --quiet

import opendatasets as od
od.download("https://www.kaggle.com/datasets/phylake1337/fire-dataset")

for root, dirs, files in os.walk("fire-dataset"):
    print(root, "->", dirs[:5], f"({len(files)} files)")

Please provide your Kaggle credentials to download this dataset. Learn more: http://bit.ly/kaggle-creds
Your Kaggle username: KGAT_88c0da93f6edca3d3193f6076c97e36c
Your Kaggle Key: ··········
Dataset URL: https://www.kaggle.com/datasets/phylake1337/fire-dataset


100%|██████████| 387M/387M [00:04<00:00, 87.2MB/s]



fire-dataset -> ['fire_dataset'] (0 files)
fire-dataset/fire_dataset -> ['fire_images', 'non_fire_images'] (0 files)
fire-dataset/fire_dataset/fire_images -> [] (755 files)
fire-dataset/fire_dataset/non_fire_images -> [] (244 files)


In [4]:
data_dir_fire = "fire-dataset/fire_dataset"

img_size = (224, 224)
batch_size = 32

train_raw_fire = tf.keras.utils.image_dataset_from_directory(
    data_dir_fire, image_size=img_size, batch_size=batch_size,
    validation_split=0.3, subset="training", seed=42, shuffle=True
)

val_test_raw_fire = tf.keras.utils.image_dataset_from_directory(
    data_dir_fire, image_size=img_size, batch_size=batch_size,
    validation_split=0.3, subset="validation", seed=42, shuffle=True
)

val_test_batches_fire = tf.data.experimental.cardinality(val_test_raw_fire)
test_raw_fire = val_test_raw_fire.take(val_test_batches_fire // 2)
val_raw_fire = val_test_raw_fire.skip(val_test_batches_fire // 2)

class_names_fire = train_raw_fire.class_names
print("Class names:", class_names_fire)

Found 999 files belonging to 2 classes.
Using 700 files for training.
Found 999 files belonging to 2 classes.
Using 299 files for validation.
Class names: ['fire_images', 'non_fire_images']


## 1B — Inspect the Data

Before doing anything else, take a real look at what you loaded. How many images does each class have? Is there anything about the data itself that might affect how you should split it, or how you should interpret your model's results later?

Write whatever code you need to answer this for yourself.

In [8]:
# Your exploration here
for classes in class_names_fire:
  print(classes, len(os.listdir(os.path.join(data_dir_fire, classes))))



fire_images 755
non_fire_images 244


*What did you notice about this dataset? Does it change how you'll approach the next steps?*



In [6]:
#There is an imblance in fire to non-fire images. Moving forward it would be best to not look
#too closely at accuracy metrics but f1 and confusion matrix instead as well as weigh non_fire images heavier.

## 1C — Build a Frozen Transfer Learning Model

Using MobileNetV2 as your pretrained base:
- Freeze the base model entirely
- Add a new output layer appropriate for this task (think about how many classes you're predicting, and what that means for your output layer's size, activation, and loss function)
- Remember the pattern from lecture for how the base model should be called

Preprocess your data (`train_raw_fire`, `val_raw_fire`, `test_raw_fire`) using MobileNetV2's expected preprocessing before building your model.

In [15]:
# Preprocess the data here
val_test_raw_fire = val_test_raw_fire.cache()
preprocess = tf.keras.applications.mobilenet_v2.preprocess_input
train_raw_fire = train_raw_fire.map(lambda x, y: (preprocess(x), y))
val_raw_fire = val_raw_fire.map(lambda x, y: (preprocess(x), y))
test_raw_fire = test_raw_fire.map(lambda x, y: (preprocess(x), y))


In [10]:
# Build your model here
base = tf.keras.applications.MobileNetV2(
    input_shape=img_size + (3,), include_top=False, weights="imagenet")
base.trainable = False

model = tf.keras.Sequential([
    base,
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dense(1, activation="sigmoid")
])


9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [11]:
# Compile your model here
model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

model.fit(train_raw_fire, validation_data=val_raw_fire, epochs=15)


Epoch 1/15
22/22 ━━━━━━━━━━━━━━━━━━━━ 61s 2s/step - accuracy: 0.7871 - loss: 0.4318 - val_accuracy: 0.8849 - val_loss: 0.2783
Epoch 2/15
22/22 ━━━━━━━━━━━━━━━━━━━━ 51s 2s/step - accuracy: 0.9614 - loss: 0.1696 - val_accuracy: 0.9496 - val_loss: 0.1438
Epoch 3/15
22/22 ━━━━━━━━━━━━━━━━━━━━ 52s 2s/step - accuracy: 0.9729 - loss: 0.1093 - val_accuracy: 0.9712 - val_loss: 0.1027
Epoch 4/15
22/22 ━━━━━━━━━━━━━━━━━━━━ 51s 2s/step - accuracy: 0.9843 - loss: 0.0868 - val_accuracy: 0.9712 - val_loss: 0.0909
Epoch 5/15
22/22 ━━━━━━━━━━━━━━━━━━━━ 92s 3s/step - accuracy: 0.9857 - loss: 0.0735 - val_accuracy: 0.9784 - val_loss: 0.0853
Epoch 6/15
22/22 ━━━━━━━━━━━━━━━━━━━━ 71s 2s/step - accuracy: 0.9857 - loss: 0.0646 - val_accuracy: 0.9568 - val_loss: 0.1082
Epoch 7/15
22/22 ━━━━━━━━━━━━━━━━━━━━ 82s 2s/step - accuracy: 0.9871 - loss: 0.0578 - val_accuracy: 0.9640 - val_loss: 0.0955
Epoch 8/15
22/22 ━━━━━━━━━━━━━━━━━━━━ 50s 2s/step - accuracy: 0.9900 - loss: 0.0528 - val_accuracy: 0.9784 - val_loss:

## 1D — Train and Evaluate

In [ ]:
# Train your model here



In [16]:
# Report validation and test accuracy here
val_loss, val_acc = model.evaluate(val_raw_fire)
test_loss, test_acc = model.evaluate(test_raw_fire)
print("Validation accuracy:", val_acc)
print("Test accuracy:", test_acc)


5/5 ━━━━━━━━━━━━━━━━━━━━ 10s 1s/step - accuracy: 0.6978 - loss: 0.7832
5/5 ━━━━━━━━━━━━━━━━━━━━ 9s 2s/step - accuracy: 0.7312 - loss: 0.7095
Validation accuracy: 0.6978417038917542
Test accuracy: 0.731249988079071


## 1E — Confusion Matrix and Discussion

Build a confusion matrix for your model's test set predictions.

**Answer:** In this context, is a false negative (missing a real fire) or a false positive (a false alarm) more costly? How might that change the decision threshold you'd actually use in practice, rather than the default 0.5?

In [18]:
# Build your confusion matrix here
y_true = np.concatenate([y for x, y in test_raw_fire], axis=0)
y_pred = (model.predict(test_raw_fire)> 0.5)

print(confusion_matrix(y_true, y_pred))

5/5 ━━━━━━━━━━━━━━━━━━━━ 9s 2s/step
[[118   0]
 [ 42   0]]


*Your answer:*
False Negatives are much worse as missing a real fire would cost lives whereas alerting to a fire that is not there would only be an inconvenience. It would be better to lower the decision threshold in order to decrease the number of false negatives, even if that risks more false positives.



---
# Part 2: Satellite Land-Use Classification (EuroSAT)

## 2A — Load the Data

In [ ]:
!wget -q "https://zenodo.org/records/7711810/files/EuroSAT_RGB.zip?download=1" -O EuroSAT_RGB.zip
!unzip -q EuroSAT_RGB.zip -d eurosat_data

data_dir_sat = "eurosat_data/EuroSAT_RGB"
print(os.listdir(data_dir_sat))

['PermanentCrop', 'Residential', 'HerbaceousVegetation', 'AnnualCrop', 'Industrial', 'Forest', 'Highway', 'Pasture', 'River', 'SeaLake']


In [ ]:
train_raw_sat = tf.keras.utils.image_dataset_from_directory(
    data_dir_sat, image_size=img_size, batch_size=batch_size,
    validation_split=0.3, subset="training", seed=42, shuffle=True
)

val_test_raw_sat = tf.keras.utils.image_dataset_from_directory(
    data_dir_sat, image_size=img_size, batch_size=batch_size,
    validation_split=0.3, subset="validation", seed=42, shuffle=True
)

val_test_batches_sat = tf.data.experimental.cardinality(val_test_raw_sat)
test_raw_sat = val_test_raw_sat.take(val_test_batches_sat // 2)
val_raw_sat = val_test_raw_sat.skip(val_test_batches_sat // 2)

class_names_sat = train_raw_sat.class_names
print("Class names:", class_names_sat)

Found 27000 files belonging to 10 classes.
Using 18900 files for training.
Found 27000 files belonging to 10 classes.
Using 8100 files for validation.
Class names: ['AnnualCrop', 'Forest', 'HerbaceousVegetation', 'Highway', 'Industrial', 'Pasture', 'PermanentCrop', 'Residential', 'River', 'SeaLake']


## 2B — Build a Frozen Transfer Learning Model

Same overall pattern as Part 1, but look closely at what's different about this task before you build your output layer, activation, and loss function.

Preprocess `train_raw_sat`, `val_raw_sat`, and `test_raw_sat` first.

In [ ]:
# Preprocess the data here



In [ ]:
# Build your model here



In [ ]:
# Compile your model here



## 2C — Train and Evaluate

In [ ]:
# Train your model here



In [ ]:
# Report validation and test accuracy here



## 2D — Confusion Matrix and Discussion

Build a confusion matrix for your model's test set predictions.

**Answer:** Which land-use classes get confused most often? Does that make visual sense to you?

In [ ]:
# Build your confusion matrix here



*Your answer:*




---
# Part 3: Compare the Two Domains

Fill in your own results:

| | Fire/Smoke | EuroSAT |
|---|---|---|
| Validation Accuracy | | |
| Test Accuracy | | |

**Answer:**
1. MobileNetV2 was originally trained on ImageNet, ordinary ground-level photos of everyday objects. Which of your two datasets do you think is the closer match to that original training, and which is the bigger mismatch? Use your actual results as evidence.
2. For each dataset, do you think there's room for improvement if you could adjust part of the pretrained model itself, rather than just the new layer on top? Would you expect that to matter more for one dataset than the other? Why?

*Your answers:*




---
# Part 4: Reflection

Answer in 4 to 6 sentences:

1. Did anything about the Fire dataset's composition affect how you approached splitting or evaluating it? Explain.
2. What's the actual difference between the output layer, activation, and loss function you used in Part 1 versus Part 2? Why does that difference exist?
3. Based on your results, does a pretrained model's usefulness depend only on how good the model is, or also on how well its original training matches your task? Explain using your own two results as evidence.

*Your answers:*



---
## Done!

Save this notebook back to your GitHub fork, then submit your two links on Canvas.